In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

In [3]:
load_dotenv()

chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [4]:
response = chat.invoke("What is 1 + 1?")
print(response.content)

1 + 1 equals 2.


In [5]:
customer_email = """
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie!
And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen.
I need your help right now, matey!
"""

style = "American English in a calm and respectful tone"

In [6]:
prompt = f"""
Translate the text delimited by triple backticks 
into a style that is {style}.

Text: ```{customer_email}```
"""

print(prompt)


Translate the text delimited by triple backticks 
into a style that is American English in a calm and respectful tone.

Text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie!
And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen.
I need your help right now, matey!
```



In [7]:
response = chat.invoke(prompt)
print(response.content)

I am quite frustrated that the lid of my blender came off and splattered my kitchen walls with smoothie. To make matters worse, the warranty does not cover the cost of cleaning up my kitchen. I would appreciate your assistance with this issue. Thank you.


In [8]:
from langchain_core.prompts import ChatPromptTemplate

In [9]:
template_string = """
Translate the text delimited by triple backticks 
into a style that is {style}.

Text: ```{text}```
"""

prompt_template = ChatPromptTemplate.from_template(template_string)

In [10]:
prompt_template.input_variables

['style', 'text']

In [11]:
customer_style = "American English in a calm and respectful tone"

customer_email = """
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie!
And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen.
I need your help right now, matey!
"""

In [12]:
customer_messages = prompt_template.format_messages(
    style=customer_style,
    text=customer_email
)

customer_messages

[HumanMessage(content="\nTranslate the text delimited by triple backticks \ninto a style that is American English in a calm and respectful tone.\n\nText: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie!\nAnd to make matters worse, the warranty don't cover the cost of cleaning up me kitchen.\nI need your help right now, matey!\n```\n", additional_kwargs={}, response_metadata={})]

In [13]:
print(customer_messages[0].content)


Translate the text delimited by triple backticks 
into a style that is American English in a calm and respectful tone.

Text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie!
And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen.
I need your help right now, matey!
```



In [14]:
customer_response = chat.invoke(customer_messages)
print(customer_response.content)

I am quite frustrated that the lid of my blender came off and splattered my kitchen walls with smoothie. To make matters worse, the warranty does not cover the cost of cleaning up my kitchen. I would appreciate your assistance with this issue. Thank you.


In [15]:
service_reply = """
Hey there customer, the warranty does not cover cleaning expenses for your kitchen 
because it was caused by misuse of the blender. 
Please make sure the lid is properly secured before using it next time.
"""

In [16]:
service_style_pirate = "a polite tone that speaks in English pirate"

In [17]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply
)


In [18]:
print(service_messages[0].content)


Translate the text delimited by triple backticks 
into a style that is a polite tone that speaks in English pirate.

Text: ```
Hey there customer, the warranty does not cover cleaning expenses for your kitchen 
because it was caused by misuse of the blender. 
Please make sure the lid is properly secured before using it next time.
```



In [19]:
service_response = chat.invoke(service_messages)
print(service_response.content)

Ahoy there, esteemed customer! I must inform ye that the warranty be not coverin' the costs of cleanin' yer galley, as it be a result of misus' of yer blender. I kindly urge ye to ensure that the lid be securely fastened before ye set sail with it again. Fair winds to ye!


In [20]:
chain = prompt_template | chat

In [21]:
response = chain.invoke({
    "style": "American English in a calm and respectful tone",
    "text": customer_email
})

print(response.content)

I am quite frustrated that the lid of my blender came off and splattered my kitchen walls with smoothie. To make matters worse, the warranty does not cover the cost of cleaning up my kitchen. I would greatly appreciate your assistance with this issue. Thank you.


In [26]:
#review info capture

In [27]:
customer_review = """
This leaf blower is pretty amazing. It has four settings: candle blower, gentle breeze,
windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present.
I think my wife liked it so much she was speechless. So far I've been the only one using it,
and I've been using it every other morning to clear the leaves on our lawn.
It is slightly more expensive than other leaf blowers out there, but I think it's worth it.
"""

In [28]:
review_template = """
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else?
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product to arrive?
If this information is not found, output -1.

price_value: Extract any sentences about the value or price.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

Text: {text}
"""

In [29]:
review_prompt_template = ChatPromptTemplate.from_template(review_template)

In [30]:
messages = review_prompt_template.format_messages(text=customer_review)
response = chat.invoke(messages)

print(response.content)

```json
{
  "gift": true,
  "delivery_days": 2,
  "price_value": "It is slightly more expensive than other leaf blowers out there, but I think it's worth it."
}
```


In [31]:
type(response.content)

str

In [32]:
from langchain_core.output_parsers import JsonOutputParser

In [33]:
parser = JsonOutputParser()

In [34]:
format_instructions = parser.get_format_instructions()
print(format_instructions)

Return a JSON object.


In [35]:
review_template_with_parser = """
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else?
Answer true if yes, false if not or unknown.

delivery_days: How many days did it take for the product to arrive?
If this information is not found, output -1.

price_value: Extract any sentences about the value or price.

Text: {text}

{format_instructions}
"""

In [36]:
review_prompt_template = ChatPromptTemplate.from_template(review_template_with_parser)

In [37]:
messages = review_prompt_template.format_messages(
    text=customer_review,
    format_instructions=format_instructions
)

print(messages[0].content)


For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else?
Answer true if yes, false if not or unknown.

delivery_days: How many days did it take for the product to arrive?
If this information is not found, output -1.

price_value: Extract any sentences about the value or price.

Text: 
This leaf blower is pretty amazing. It has four settings: candle blower, gentle breeze,
windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present.
I think my wife liked it so much she was speechless. So far I've been the only one using it,
and I've been using it every other morning to clear the leaves on our lawn.
It is slightly more expensive than other leaf blowers out there, but I think it's worth it.


Return a JSON object.



In [38]:
response = chat.invoke(messages)
print(response.content)

```json
{
  "gift": true,
  "delivery_days": 2,
  "price_value": "It is slightly more expensive than other leaf blowers out there, but I think it's worth it."
}
```


In [39]:
output_dict = parser.parse(response.content)
output_dict

{'gift': True,
 'delivery_days': 2,
 'price_value': "It is slightly more expensive than other leaf blowers out there, but I think it's worth it."}

In [40]:
type(output_dict)

dict

In [41]:
print(output_dict["gift"])
print(output_dict["delivery_days"])
print(output_dict["price_value"])

True
2
It is slightly more expensive than other leaf blowers out there, but I think it's worth it.


In [43]:
review_chain = review_prompt_template | chat | parser

In [44]:
result = review_chain.invoke({
    "text": customer_review,
    "format_instructions": format_instructions
})
result

{'gift': True,
 'delivery_days': 2,
 'price_value': "It is slightly more expensive than other leaf blowers out there, but I think it's worth it."}